# Design: `/configure` settings browser

**Status:** Written — Approach 1 approved 2026-08-21; formal cells verified; awaiting user spec review  
**Date:** 2026-08-21  
**Design epic:** `bd-ma96` (`spur:plan-id:7f3c2a91-4e18-4b6d-9c0a-2e8f1b5d6a70`)  
**Supersedes Phase-2 deferral in:** `docs/superpowers/specs/2026-07-06-configure-agent-settings-browser-design.md`  
**NS-Mermaid profile:** `relational_lia` (`flowchart`) **v1**, engine `z3` / `qf_lia_bool_int_enum`, binding `inline_annotations`

## Decision

`/configure` becomes a **sectioned settings browser** over a curated subset of `.spur/config.toml`, not an agent-only view. The command stays exclusive (TUI-owned). The existing agent pane remains the default section; Graph, TUI, and Skills join it as sibling sections.

**Chosen approach:** one view with a left-pane section list (`agents | graph | tui | skills`) and a right-pane field editor. Persist through `spur_acp::config::update_config`. **Persist-then-apply** is the save contract for every section (`SAVE-APPLY`).

## Why this decision

Phase 1 shipped `AgentConfigBrowserView` and left Phase 2 as “fold `/theme` / `/vim` / `disable_paste_burst` later.” `SpurConfig` now also owns `graph.embedding_model` and `skills.projection_mode` — both enumerable, both currently file-edit only. A second command or a dump of every `SpurConfig` table would either fragment UX or expose identity/runtime knobs that must stay restart-only.

This spec therefore:

1. Reuses the Phase 1 browser chrome instead of a new slash command.
2. Adds only closed or already-curated knobs (solver-bound enums + existing agent fields + TUI prefs).
3. Leaves brain, cost, pm, loops, worktree, failover, and agent identity (`name` / `command` / `transport` / `kind`) out of live edit.

## Goals

- One `/configure` surface for Agents (existing curated fields), Graph (`embedding_model`), TUI (`edit_mode`, `theme`, `disable_paste_burst`), and Skills (`projection_mode`).
- Section focus is a total function of the selected section (`CONFIGURE-SECTION`): exactly one of `agents | graph | tui | skills` is focused.
- Enumerable knobs are closed sets already parsed in code (`GRAPH-EMBEDDING`, `SKILLS-PROJECTION`, `TUI-EDIT-MODE`). Unknown values are rejected in the view before persist.
- Save is **persist-then-apply** (`SAVE-APPLY`): if `update_config` fails, in-memory state is not treated as saved. Phase 1 currently mutates `App.config.agents` *before* the orchestrator persist; this spec corrects that inversion for the unified save path.
- `/theme` and `/vim` remain shortcuts; they write the same `TuiConfig` fields. `/configure` is the browser, not a replacement for those commands.
- Command copy updates from “Open agent settings browser” to “Open settings browser”. Hint: `[section|agent-name]`.

## Non-goals

- Editing `brain`, `failover`, `worktree`, `cost`, `pm`, `bot`, `plan`, `delegation`, `log`, `context_service`, or `peer_mailbox_enabled`.
- Add/remove agents, or live-edit agent `name` / `command` / `transport` / `kind`.
- File-watcher hot-reload of hand edits or `spur config set` (unchanged from Phase 1).
- Reloading an already-loaded embedding `OnceLock` or already-spawned worker ACP session.
- Editing `skills.bundled_dir` (path, not an enum).
- A new slash command (`/settings`). `/configure` is the name.
- CAS / merge of concurrent external TOML writers (`update_config` remains last-rename-wins, same as `/theme`).

## Formal gate: section focus (`CONFIGURE-SECTION`)

The left pane lists four sections. Focus is a total, deterministic, exclusive function of `selected`. Implementation must not allow two sections focused, or none focused, for any input in the enum.

`/configure` with no args focuses `agents`. `/configure graph|tui|skills` focuses that section. `/configure <agent-name>` focuses `agents` and preselects the matching `AgentConfig` (Phase 1 behavior).

In [ ]:
flowchart TD
    SPEC["`@spec CONFIGURE-SECTION
@type Section = enum[agents, graph, tui, skills]
@input selected: Section
@output focused: Section`"]
    AGENTS["`@branch AGENTS
@when selected = agents
@ensures AGENTS_FOCUSED: focused = agents`"]
    GRAPH["`@branch GRAPH
@when selected = graph
@ensures GRAPH_FOCUSED: focused = graph`"]
    TUI["`@branch TUI
@when selected = tui
@ensures TUI_FOCUSED: focused = tui`"]
    SKILLS["`@branch SKILLS
@when selected = skills
@ensures SKILLS_FOCUSED: focused = skills`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify CONSIST: witness consistency`"]
    SPEC --> AGENTS --> CHECK
    SPEC --> GRAPH --> CHECK
    SPEC --> TUI --> CHECK
    SPEC --> SKILLS --> CHECK

## Architecture

**View.** Keep `ViewId::AgentConfigBrowser` as the navigation target in this epic (rename is optional and not required). Internally the view grows a `ConfigureSection` enum matching `CONFIGURE-SECTION`. Agents pane reuses the existing left-agent-list + field editor. Other sections show a single field list (no nested agent list).

**Command.** `SpurLocalSource` entry `configure` stays exclusive. Description: “Open settings browser”. Args:

| Arg | Focus |
|---|---|
| (none) | `agents` |
| `graph` / `tui` / `skills` | that section |
| other token | `agents` + Phase 1 agent preselect |

**Save path.** One new (or generalized) `UserInput` / `InteractiveInput` family, not four independent persist pipes:

1. View validates the draft (enum parse / existing `sanitize_agent_additional_directories`).
2. Orchestrator calls `update_config(&config_path, |c| { …section mutate… })`.
3. Only on `Ok` does live-apply run (TUI `self.config` + section-specific in-process hooks).
4. Confirmation or error event flashes in the view.

Phase 1 `Action::AgentConfigSaveRequested` must move onto this persist-then-apply path so agents cannot diverge from disk on write failure.

**Layered config.** Writes still target the **project** `.spur/config.toml` that `update_config` already uses (`/theme` today). User-level `~/.spur/config.toml` is not edited by `/configure`. `SPUR_EMBEDDING_MODEL` continues to override `graph.embedding_model` at read time; the browser shows the configured TOML value and a read-only note when the env override is set.

## Editable fields

### Agents (unchanged curated subset)

`additional_directories`, `args`, `capabilities`, `skip_permissions` + `skip_permissions_args` + `skip_permissions_session_mode`, `profile`. Identity fields remain read-only.

### Graph

`graph.embedding_model` — closed aliases stored as TOML strings `nomic` | `coderank` | `jina-code`. Formal enum uses `jina_code` (hyphens are not NS-Mermaid identifiers). Parser already accepts those aliases via `EmbeddingModelSelection::parse`.

### TUI

| Field | Domain | Live-apply |
|---|---|---|
| `tui.edit_mode` | `emacs` \| `vim` (`EditorMode`) | Yes — same in-memory `App.edit_mode` as `/vim`, but now persisted |
| `tui.theme` | discovered names (built-in + `.spur/themes` + `~/.spur/themes`), not a closed solver enum | Yes — existing `/theme` apply path |
| `tui.disable_paste_burst` | `bool` | Yes — in-process flag used by paste handling |

### Skills

`skills.projection_mode` — `catalog_only` \| `all_active` (`SkillsProjectionMode`, serde `snake_case`). Applies to **newly reconciled** brain/worker sessions, not already-projected sessions.

## Formal gate: graph embedding model (`GRAPH-EMBEDDING`)

Picker values are exactly `{nomic, coderank, jina_code}`. The TOML string for `jina_code` is `jina-code`. Free-text Hugging Face ids are **not** offered in the TUI (they remain env/TOML-parseable if added later; `/configure` only writes the three aliases). Changing the model persists immediately; the process-wide embedder `OnceLock` is **not** swapped. Next `spur tui` / indexer start observes the new model. Show a hint: “takes effect on next embedding load (restart).”

In [ ]:
flowchart TD
    SPEC["`@spec GRAPH-EMBEDDING
@type Model = enum[nomic, coderank, jina_code]
@input model: Model
@output chosen: Model`"]
    NOMIC["`@branch NOMIC
@when model = nomic
@ensures NOMIC_CHOSEN: chosen = nomic`"]
    CR["`@branch CODERANK
@when model = coderank
@ensures CR_CHOSEN: chosen = coderank`"]
    JINA["`@branch JINA
@when model = jina_code
@ensures JINA_CHOSEN: chosen = jina_code`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify CONSIST: witness consistency`"]
    SPEC --> NOMIC --> CHECK
    SPEC --> CR --> CHECK
    SPEC --> JINA --> CHECK

## Formal gate: skills projection (`SKILLS-PROJECTION`)

`skills.projection_mode` is a total function over `{catalog_only, all_active}`. Default remains `catalog_only`. Changing it persists; already-running sessions keep their projected skill set. Hint: “applies to newly reconciled sessions.”

In [ ]:
flowchart TD
    SPEC["`@spec SKILLS-PROJECTION
@type Mode = enum[catalog_only, all_active]
@input mode: Mode
@output chosen: Mode`"]
    CAT["`@branch CATALOG
@when mode = catalog_only
@ensures CAT_CHOSEN: chosen = catalog_only`"]
    ALL["`@branch ALL_ACTIVE
@when mode = all_active
@ensures ALL_CHOSEN: chosen = all_active`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify CONSIST: witness consistency`"]
    SPEC --> CAT --> CHECK
    SPEC --> ALL --> CHECK

## Formal gate: TUI edit mode (`TUI-EDIT-MODE`)

`tui.edit_mode` is `{emacs, vim}` (`EditorMode`, serde lowercase). Theme is **not** in this gate: it is an open discovered-name picker. `disable_paste_burst` is a bool toggle (no partition proof). `/vim` continues to toggle `edit_mode` live; after this epic it must also persist via `update_config` so `/configure` and `/vim` cannot disagree on disk.

In [ ]:
flowchart TD
    SPEC["`@spec TUI-EDIT-MODE
@type Mode = enum[emacs, vim]
@input mode: Mode
@output chosen: Mode`"]
    EMACS["`@branch EMACS
@when mode = emacs
@ensures EMACS_CHOSEN: chosen = emacs`"]
    VIM["`@branch VIM
@when mode = vim
@ensures VIM_CHOSEN: chosen = vim`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify CONSIST: witness consistency`"]
    SPEC --> EMACS --> CHECK
    SPEC --> VIM --> CHECK

## Formal gate: persist-then-apply (`SAVE-APPLY`)

`applied = yes` if and only if persist succeeded. Persist failure ⇒ `applied = no` (flash error; drafts stay in the view). Never apply-then-persist.

`Applied` is an enum (`yes|no`) rather than `Bool` so the relation is an enum output (Bool bi-implication on `@requires` is rejected by this profile when mixed with branch `@ensures`).

### Live-apply matrix

| Section / field | Persist | Live-apply on success |
|---|---|---|
| Agents curated fields | `update_config` agents.entries | Yes — existing `agent_configs` `RwLock`; next delegation |
| `graph.embedding_model` | `update_config` graph | No process-wide embedder swap; hint restart |
| `tui.edit_mode` | `update_config` tui | Yes — `App.edit_mode` |
| `tui.theme` | `update_config` tui | Yes — existing theme loader |
| `tui.disable_paste_burst` | `update_config` tui | Yes — in-process paste-burst flag |
| `skills.projection_mode` | `update_config` skills | No already-running session reprojection; next reconcile |

`/vim` after this epic: persist then apply (today it is apply-only). `/theme` already persist+apply; keep that order.

In [ ]:
flowchart TD
    SPEC["`@spec SAVE-APPLY
@type Applied = enum[yes, no]
@input persist_ok: Bool
@output applied: Applied`"]
    OK["`@branch PERSIST_OK
@when persist_ok = true
@ensures APPLIED: applied = yes`"]
    FAIL["`@branch PERSIST_FAIL
@when persist_ok = false
@ensures NOT_APPLIED: applied = no`"]
    CHECK["`@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify CONSIST: witness consistency`"]
    SPEC --> OK --> CHECK
    SPEC --> FAIL --> CHECK

## Error handling

- **Unknown enum / unknown theme:** field-level validation in the view; do not dispatch persist.
- **Persist failure:** flash `update_config` error; `applied = no`; draft remains.
- **Env override (`SPUR_EMBEDDING_MODEL`):** Graph pane still edits TOML; banner that env wins at read time.
- **Concurrent hand-edit:** last-rename-wins, same as `/theme`.
- **Comment stripping:** full `SpurConfig` serde round-trip already drops TOML comments; accepted pre-existing tradeoff.

## Testing (TDD)

- View: section focus (`CONFIGURE-SECTION`) — one focused section for each of the four inputs; arg routing (`graph` vs agent name).
- View: embedding picker rejects unknown strings; writes `nomic` / `coderank` / `jina-code` only.
- View: projection mode round-trips `catalog_only` / `all_active`.
- View: `edit_mode` emacs/vim; theme picker lists discovered names only.
- Persist: temp `.spur/config.toml` — each section mutate preserves sibling substructs (`update_config_preserves_all_sibling_substructs` style).
- Persist-then-apply: when `update_config` is stubbed to fail, `App.config` and `agent_configs` lock are unchanged.
- Live-apply: TUI `edit_mode` / theme / paste-burst change without restart; agents still reach next delegation (existing Phase 1 test).
- Non-apply: embedding model change does **not** require swapping `OnceLock` in-process (assert persist only).

## Task decomposition (for writing-plans)

Wide DAG after a shared view-shell + save-path contract:

1. **View shell + `ConfigureSection`** (binds `CONFIGURE-SECTION`) — command args, left pane, keep agents pane working.
2. **Unified persist path** (binds `SAVE-APPLY`) — generalize `UpdateAgentConfig` to section patches; fix Phase 1 apply-before-persist.
3. **Graph pane** (binds `GRAPH-EMBEDDING`) — can start after (1); persist after (2).
4. **TUI pane** (binds `TUI-EDIT-MODE` + theme/paste) — parallel with (3) after (1); persist after (2). Also persist `/vim`.
5. **Skills pane** (binds `SKILLS-PROJECTION`) — parallel with (3)/(4).
6. **Docs + command copy** — user-docs configuration page; `/configure` help text.

## Risks and migration

- **Phase 1 inversion:** moving agent save to persist-then-apply can add a flash of “not saved” if disk fails — that is the point. Optimistic UI drafts stay local until Ok.
- **`/vim` behavior change:** currently live-only. After this epic it persists. Document in user-docs; no schema change.
- **Embedding cache:** model files already live under `~/.spur/cache/fastembed`. Switching models does not delete the previous cache.
- **Skills projection:** `all_active` is a large context change for new sessions; the pane should show a one-line consequence, not a confirmation modal.

## Approaches considered (recorded)

1. **Sectioned `/configure` browser** (chosen) — reuses Phase 1 chrome; four sections; persist-then-apply.
2. **Separate slash commands** (`/graph-config`, `/skills-config`) — fragments discovery; rejected.
3. **Full `SpurConfig` tree editor** — exposes identity and runtime foot-guns; rejected for this epic.

## Proof evidence

All five formal cells ran 2026-08-21. Profile `relational_lia` v1. Each cell: `verified=true`, `proof_fresh=true`, 4/4 obligations matched (`CONSIST` sat, `COVER`/`DET`/`EXCL` unsat).

| `@spec` | Cell | source_hash | ir_hash (preflight content_hash) | report_hash |
|---|---|---|---|---|
| `CONFIGURE-SECTION` | `5e1c0a12-…9e04` | `79671f19fccfbb8a1db73fd03cc9d86d2d74a15de4fe65c0a02c931043f858ce` | `4dac9430569f2bf867e25055329dc4cf21b11fb8ae87b52cbb5ae03366f29816` | `3ee8807692d46d4da7d02a827059402f4152039f7591500820c413ebc2dca478` |
| `GRAPH-EMBEDDING` | `5e1c0a12-…9e07` | `72f8d1c22376b08a531565c866c022c3190714e1ccad670cf7b4e5063b2a584a` | `e64932a2cd652e316ca94da85d432682ab506e0341d64cbd151dd00da8590578` | `798e2ec7e3e8de32c7d6880fb5e7ca86d12acf46591c092c20ab63a32b27bb09` |
| `SKILLS-PROJECTION` | `5e1c0a12-…9e09` | `7defa71e4d05715cd9678318baa477c4ef2ef6504886c3f727a32f0e4614a69f` | `10a33cfcde736c21bec80cb5f0e5fa8ce61e6bb20733a29f20dfe85ca55fcc7e` | `310899e6ba647f3a79bfc0d80d9475c2aa8f3301a6cad2d1682bc3a4b598b33c` |
| `TUI-EDIT-MODE` | `5e1c0a12-…9e0b` | `3c2f62783cb3c525daca865418b74ab77685e127dbb4252bcaf0dd30d718702a` | `329eafb7c0f61abc9d4686604861683853d1c3e587ef93961b626a2e9bd2d2b3` | `82d116bb56bc53e070f49c9fc672c9de0d445635c30f74aa8eba718b06006151` |
| `SAVE-APPLY` | `5e1c0a12-…9e0d` | `d33cf7e70c36c28da63f0382ab767d62df9ea747fd6e5d1eab103a50217010b7` | `15f89dde8b630fc9e1d2c2f350e96ef5ad3a85cfd0f8127a6568948948e2418b` | `ccd5a6a70a009113ddf0aa4ae395012f27c98956dd9e947c65c7550c8398fbfd` |

Independent catalog checks from design: `configuration.selection_cardinality` sat `sol_6ed75de560f74927` (one section) / unsat `sol_3c9e0fd0bd7046ab` (two sections).